#Import modules

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

#Data for metabolic genes

In [3]:
rna = pd.read_csv('drive/My Drive/Colab Notebooks/Deep_learning_metabolomics_prediction/TCGA_Pan/TCGA_Pan_TPM.csv', index_col = 0)
rna.index.name = None
rna = rna.T
rna.head(3)

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000288661,ENSG00000288662,ENSG00000288663,ENSG00000288665,ENSG00000288667,ENSG00000288669,ENSG00000288670,ENSG00000288671,ENSG00000288674,ENSG00000288675
TCGA-D8-A146-01A-31R-A115-07,49.6341,9.3826,115.1737,20.1202,6.1859,10.9585,30.9154,47.9991,19.9032,56.4289,...,0.0,0.0,0.4513,0.0,0.0,0.0,10.5835,0.0,0.0411,1.1776
TCGA-AQ-A0Y5-01A-11R-A14M-07,12.0296,0.3785,134.9047,15.5758,4.3777,4.0015,29.7787,91.9948,15.5230,51.4084,...,0.0,0.0,0.1525,0.0,0.0,0.0,26.3496,0.0,0.0387,1.3672
TCGA-C8-A274-01A-11R-A16F-07,90.4249,0.0000,110.8229,31.8373,15.4869,3.3594,6.9632,33.7734,9.6288,61.4116,...,0.0,0.0,0.3309,0.0,0.0,0.0,25.0229,0.0,0.0669,0.5202


In [4]:
variances = rna.var()
max_var_col = variances.idxmax()
print(max_var_col)

ENSG00000211592


In [5]:
met_genes = pd.read_csv('drive/My Drive/Colab Notebooks/GP_lab_data/FBA_metabolic_atlas_models/human_gem_associated_genes.csv')
met_genes

,gene_id,Symbol
0,ENSG00000198712,MT-CO2
1,ENSG00000198804,MT-CO1
2,ENSG00000198938,MT-CO3
3,ENSG00000198899,MT-ATP6
4,ENSG00000198886,MT-ND4
...,...,...
2879,ENSG00000167751,KLK2
2880,ENSG00000129873,CDY2B
2881,ENSG00000182415,CDY2A
2882,ENSG00000172352,CDY1B


In [6]:
#Make sure the genes are the same as in the training data for the transfer model

transfer_data = pd.read_csv('drive/My Drive/Colab Notebooks/Deep_learning_metabolomics_prediction/gene_expression_benedetti_dataset.csv', index_col = 0)
transfer_data.head(3)

,ENSG00000121410,ENSG00000175899,ENSG00000291190,ENSG00000171428,ENSG00000156006,ENSG00000253937,ENSG00000196136,ENSG00000114771,ENSG00000127837,ENSG00000129673,...,ENSG00000207734,ENSG00000207735,ENSG00000207925,ENSG00000283330,ENSG00000207699,ENSG00000291012,ENSG00000274847,ENSG00000261509,ENSG00000276941,ENSG00000183385
MSKC-00397,1.304026,320.190822,4.301926,8.314472,0.964676,NaN,123.587134,0.210955,180.621121,8.316147,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MSKC-00399,1.928762,599.699496,1.251719,10.039836,3.234161,NaN,3.693906,0.000000,227.394412,0.349771,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MSKC-00400,0.869958,759.630310,1.620558,5.763073,0.369455,NaN,0.519229,0.242377,219.065986,0.769822,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
transfer_data_met = transfer_data.reindex(met_genes['gene_id'].to_list(), axis = 'columns')
transfer_data_met = pd.concat([transfer_data[['ENSG00000211592']], transfer_data_met], axis = 'columns')
transfer_data_met = transfer_data_met.dropna(how = 'all', axis = 'columns')
transfer_data_met = transfer_data_met.fillna(0)
transfer_data_met.head(3)

,ENSG00000211592,ENSG00000198712,ENSG00000198804,ENSG00000198938,ENSG00000198899,ENSG00000198886,ENSG00000212907,ENSG00000198763,ENSG00000198840,ENSG00000156508,...,ENSG00000205923,ENSG00000183747,ENSG00000066813,ENSG00000182601,ENSG00000156885,ENSG00000182156,ENSG00000166948,ENSG00000197838,ENSG00000167751,ENSG00000182415
MSKC-00397,3046.071614,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2929.049654,...,19.356151,5.898651,8.225321,0.026444,0.384994,2.161147,0.0,0.0,0.127099,0.0
MSKC-00399,386.607578,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3537.456407,...,23.797005,173.680890,235.708998,0.000000,4.840216,4.216679,0.0,0.0,0.000000,0.0
MSKC-00400,109.967242,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2044.849531,...,23.372477,534.686682,411.479407,0.000000,0.000000,0.316985,0.0,0.0,0.000000,0.0


In [8]:
transfer_data_met.to_csv('drive/My Drive/Colab Notebooks/Deep_learning_metabolomics_prediction/transfer_model_rna_met.csv', index = True)

In [9]:
rna_met = rna.reindex(transfer_data_met.columns, axis = 'columns')
rna_met = pd.concat([rna[['ENSG00000211592']], rna_met], axis = 'columns')
rna_met = rna_met.fillna(0)
rna_met.head(3)

,ENSG00000211592,ENSG00000211592,ENSG00000198712,ENSG00000198804,ENSG00000198938,ENSG00000198899,ENSG00000198886,ENSG00000212907,ENSG00000198763,ENSG00000198840,...,ENSG00000205923,ENSG00000183747,ENSG00000066813,ENSG00000182601,ENSG00000156885,ENSG00000182156,ENSG00000166948,ENSG00000197838,ENSG00000167751,ENSG00000182415
TCGA-D8-A146-01A-31R-A115-07,7654.8063,7654.8063,24344.3785,16191.8522,19880.8536,13343.2173,17287.6749,5208.1979,12236.5064,12141.7301,...,0.0,0.0402,0.0162,0.1149,0.1394,0.0562,0.0,0.0749,0.0831,0.0
TCGA-AQ-A0Y5-01A-11R-A14M-07,10.8013,10.8013,15466.3528,22624.9765,22247.2836,12629.4103,18353.2198,7174.4777,9078.7267,7283.1951,...,0.0,0.0000,0.0152,0.0902,0.0000,0.0000,0.0,0.0000,0.1173,0.0
TCGA-C8-A274-01A-11R-A16F-07,216.4474,216.4474,21705.3843,38997.7958,44549.4302,19402.3325,26656.2610,10667.9069,12433.0264,8790.3434,...,0.0,0.0000,0.0056,0.0000,0.0000,0.0131,0.0,0.6273,0.0290,0.0


In [10]:
rna_met.to_csv('drive/My Drive/Colab Notebooks/Deep_learning_metabolomics_prediction/TCGA_Pan_rna_met.csv', index = True)

In [11]:
annot = pd.read_csv('drive/My Drive/Colab Notebooks/Deep_learning_metabolomics_prediction/TCGA_Pan/TCGA_Samples_Tissue_Type.csv', index_col = 0)
annot.index.name = None
annot.head(3)

,patient_barcorde,sample_type,samples_tissue_type
TCGA-02-0047-01A-01R-1849-01,TCGA-02-0047,Tumour,TCGA-GBM
TCGA-02-0055-01A-01R-1849-01,TCGA-02-0055,Tumour,TCGA-GBM
TCGA-02-2483-01A-01R-1849-01,TCGA-02-2483,Tumour,TCGA-GBM


In [12]:
print(rna_met.shape)
print(annot.shape)

(10668, 2856)
(10668, 3)


In [13]:
rna_met_annot = pd.concat([annot[['sample_type']], rna[[max_var_col]], rna_met], axis = 'columns')
rna_met_annot.head(3)

,sample_type,ENSG00000211592,ENSG00000211592,ENSG00000211592,ENSG00000198712,ENSG00000198804,ENSG00000198938,ENSG00000198899,ENSG00000198886,ENSG00000212907,...,ENSG00000205923,ENSG00000183747,ENSG00000066813,ENSG00000182601,ENSG00000156885,ENSG00000182156,ENSG00000166948,ENSG00000197838,ENSG00000167751,ENSG00000182415
TCGA-02-0047-01A-01R-1849-01,Tumour,59.9245,59.9245,59.9245,21699.5817,27668.9818,20446.0504,18985.4760,28133.5996,12051.6972,...,0.0,0.0587,0.0000,1.6474,0.2032,0.1911,0.0000,0.0000,0.0000,0.0
TCGA-02-0055-01A-01R-1849-01,Tumour,858.6550,858.6550,858.6550,12783.1665,11524.2146,10867.5278,6889.9482,9928.6039,2621.6115,...,0.0,0.0000,0.0000,5.2644,0.3234,0.0000,0.0665,0.0000,0.0120,0.0
TCGA-02-2483-01A-01R-1849-01,Tumour,61.6183,61.6183,61.6183,24481.8793,18086.7576,24786.5198,18700.0518,16230.6300,5326.0850,...,0.0,0.0000,0.0079,0.9862,0.2709,0.0910,0.0279,0.0364,0.0101,0.0


In [14]:
rna_met_annot.to_csv('drive/My Drive/Colab Notebooks/Deep_learning_metabolomics_prediction/TCGA_Pan_rna_met_annot.csv', index = True)

#Unbiased feature selection

In [ ]:
rna.head(3)

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000288661,ENSG00000288662,ENSG00000288663,ENSG00000288665,ENSG00000288667,ENSG00000288669,ENSG00000288670,ENSG00000288671,ENSG00000288674,ENSG00000288675
TCGA-D8-A146-01A-31R-A115-07,49.6341,9.3826,115.1737,20.1202,6.1859,10.9585,30.9154,47.9991,19.9032,56.4289,...,0.0,0.0,0.4513,0.0,0.0,0.0,10.5835,0.0,0.0411,1.1776
TCGA-AQ-A0Y5-01A-11R-A14M-07,12.0296,0.3785,134.9047,15.5758,4.3777,4.0015,29.7787,91.9948,15.5230,51.4084,...,0.0,0.0,0.1525,0.0,0.0,0.0,26.3496,0.0,0.0387,1.3672
TCGA-C8-A274-01A-11R-A16F-07,90.4249,0.0000,110.8229,31.8373,15.4869,3.3594,6.9632,33.7734,9.6288,61.4116,...,0.0,0.0,0.3309,0.0,0.0,0.0,25.0229,0.0,0.0669,0.5202


In [ ]:
transfer_data.head(3)

,ENSG00000121410,ENSG00000175899,ENSG00000291190,ENSG00000171428,ENSG00000156006,ENSG00000253937,ENSG00000196136,ENSG00000114771,ENSG00000127837,ENSG00000129673,...,ENSG00000207734,ENSG00000207735,ENSG00000207925,ENSG00000283330,ENSG00000207699,ENSG00000291012,ENSG00000274847,ENSG00000261509,ENSG00000276941,ENSG00000183385
MSKC-00397,1.304026,320.190822,4.301926,8.314472,0.964676,NaN,123.587134,0.210955,180.621121,8.316147,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MSKC-00399,1.928762,599.699496,1.251719,10.039836,3.234161,NaN,3.693906,0.000000,227.394412,0.349771,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MSKC-00400,0.869958,759.630310,1.620558,5.763073,0.369455,NaN,0.519229,0.242377,219.065986,0.769822,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
variances_top = rna.var()
threshold = variances_top.quantile(0.95)
rna_top = rna.loc[:, variances_top >= threshold]
rna_top.shape

(10668, 3031)

In [ ]:
rna_top.head(3)

,ENSG00000000005,ENSG00000000419,ENSG00000000971,ENSG00000001617,ENSG00000002549,ENSG00000002586,ENSG00000002726,ENSG00000002834,ENSG00000002933,ENSG00000003989,...,ENSG00000281383,ENSG00000281990,ENSG00000282122,ENSG00000282431,ENSG00000282639,ENSG00000282651,ENSG00000283475,ENSG00000283498,ENSG00000283907,ENSG00000286190
TCGA-D8-A146-01A-31R-A115-07,9.3826,115.1737,30.9154,89.2583,96.8896,200.3364,3.1527,159.0237,64.0686,16.7347,...,2.8579,17.9344,62.3372,5.4955,68.7485,110.661,45.7744,37.2402,0.5340,3.9128
TCGA-AQ-A0Y5-01A-11R-A14M-07,0.3785,134.9047,29.7787,74.5808,66.8669,176.3045,17.2985,162.9267,80.5922,376.5188,...,1.1434,0.0000,0.0000,0.0000,0.0000,0.000,16.0672,33.5950,0.7540,3.4764
TCGA-C8-A274-01A-11R-A16F-07,0.0000,110.8229,6.9632,66.8816,100.7954,71.8287,0.1458,179.6223,15.9810,94.7782,...,1.2957,0.0000,8.1241,0.0000,0.0000,0.000,107.6903,46.5395,0.6518,1.1040


In [ ]:
transfer_data_top = transfer_data.reindex(rna_top.columns.to_list(), axis = 'columns')
transfer_data_top.shape

(443, 3031)

In [ ]:
transfer_data_top.head(3)

,ENSG00000000005,ENSG00000000419,ENSG00000000971,ENSG00000001617,ENSG00000002549,ENSG00000002586,ENSG00000002726,ENSG00000002834,ENSG00000002933,ENSG00000003989,...,ENSG00000281383,ENSG00000281990,ENSG00000282122,ENSG00000282431,ENSG00000282639,ENSG00000282651,ENSG00000283475,ENSG00000283498,ENSG00000283907,ENSG00000286190
MSKC-00397,0.083832,119.085084,122.411217,33.512536,212.796414,338.895524,10.487695,153.242942,421.044928,27.855413,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.756917,NaN,NaN
MSKC-00399,0.991959,102.072551,25.971543,37.192402,132.834170,180.577386,97.414163,108.454827,420.242464,12.345781,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32.082926,NaN,NaN
MSKC-00400,0.642128,121.438296,8.719379,90.323070,92.971536,240.921228,11.744602,172.203495,2656.460755,12.439999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.208051,NaN,NaN


In [ ]:
rna_top.to_csv('drive/My Drive/Colab Notebooks/Deep_learning_metabolomics_prediction/TCGA_Pan_rna_top_variance_genes.csv', index = True)
transfer_data_top.to_csv('drive/My Drive/Colab Notebooks/Deep_learning_metabolomics_prediction/transfer_data_top_variance_genes.csv', index = True)

#Preparing transcription factor dataset

In [ ]:
tf = pd.read_csv('drive/My Drive/Colab Notebooks/Human_TFs_DatabaseExtract_v_1.01.csv', index_col = 0)
tf.head(3)

,Ensembl ID,HGNC symbol,DBD,Is TF?,TF assessment,Binding mode,Motif status,Final Notes,Final Comments,Interpro ID(s),...,CisBP considers it a TF?,TFCat classification,Is a GO TF?,Initial assessment,Curator 1,Curator 2,TFclass considers it a TF?,Go Evidence,Pfam Domains (By ENSP ID),Is C2H2 ZF(KRAB)?
0,ENSG00000137203,TFAP2A,AP-2,Yes,Known motif,Monomer or homomultimer,High-throughput in vitro,NaN,NaN,IPR008121;IPR013854,...,Yes,TF Gene_DNA-Binding: sequence-specific_DNA Bin...,Yes,"1a1, Direct HQ evidence",Sam Lambert,Yimeng Yin,Yes,$#ENSG00000137203#GO:0000981#sequence-specific...,$#ENSP00000368928#ENSG00000137203#ENST00000379...,False
1,ENSG00000008196,TFAP2B,AP-2,Yes,Known motif,Monomer or homomultimer,High-throughput in vitro,NaN,NaN,IPR008122;IPR013854,...,Yes,TF Gene_DNA-Binding: sequence-specific_DNA Bin...,Yes,"1a1, Direct HQ evidence",Matt Weirauch,Yimeng Yin,Yes,$#ENSG00000008196#GO:0000981#sequence-specific...,$#ENSP00000377265#ENSG00000008196#ENST00000393...,False
2,ENSG00000087510,TFAP2C,AP-2,Yes,Known motif,Monomer or homomultimer,High-throughput in vitro,NaN,NaN,IPR008123;IPR013854,...,Yes,No,Yes,"1a1, Direct HQ evidence",Matt Weirauch,Yimeng Yin,Yes,$#ENSG00000087510#GO:0001077#RNA polymerase II...,$#ENSP00000201031#ENSG00000087510#ENST00000201...,False


In [ ]:
common_tf_rna = list(set(tf['Ensembl ID'].to_list()).intersection(rna.columns.to_list()))
print(common_tf_rna)

['ENSG00000101096', 'ENSG00000145736', 'ENSG00000164930', 'ENSG00000213988', 'ENSG00000220201', 'ENSG00000175213', 'ENSG00000061273', 'ENSG00000151748', 'ENSG00000171116', 'ENSG00000124226', 'ENSG00000159184', 'ENSG00000112592', 'ENSG00000177125', 'ENSG00000176371', 'ENSG00000137947', 'ENSG00000130684', 'ENSG00000177463', 'ENSG00000196323', 'ENSG00000234444', 'ENSG00000146463', 'ENSG00000174332', 'ENSG00000183770', 'ENSG00000117000', 'ENSG00000131127', 'ENSG00000163558', 'ENSG00000135111', 'ENSG00000179943', 'ENSG00000111206', 'ENSG00000198105', 'ENSG00000052850', 'ENSG00000172534', 'ENSG00000161265', 'ENSG00000160199', 'ENSG00000165030', 'ENSG00000159256', 'ENSG00000182463', 'ENSG00000120693', 'ENSG00000166173', 'ENSG00000064933', 'ENSG00000129028', 'ENSG00000187801', 'ENSG00000165702', 'ENSG00000136630', 'ENSG00000170374', 'ENSG00000135363', 'ENSG00000180818', 'ENSG00000268738', 'ENSG00000127989', 'ENSG00000105939', 'ENSG00000125533', 'ENSG00000249471', 'ENSG00000165061', 'ENSG000001

In [ ]:
rna_tf = rna[common_tf_rna]
rna_tf.head(3)

,ENSG00000101096,ENSG00000145736,ENSG00000164930,ENSG00000213988,ENSG00000220201,ENSG00000175213,ENSG00000061273,ENSG00000151748,ENSG00000171116,ENSG00000124226,...,ENSG00000165119,ENSG00000124160,ENSG00000124191,ENSG00000182732,ENSG00000125520,ENSG00000197951,ENSG00000189042,ENSG00000114861,ENSG00000237440,ENSG00000120094
TCGA-D8-A146-01A-31R-A115-07,10.9194,5.0623,40.8331,0.8124,0.8952,19.3743,34.1488,36.9911,0.1008,81.6687,...,489.2058,100.0648,2.5685,0.3759,118.9051,8.9125,7.5243,9.1896,16.7620,0.5040
TCGA-AQ-A0Y5-01A-11R-A14M-07,8.0088,0.2647,53.8007,2.1311,0.3320,10.1802,19.9714,21.6635,0.0000,45.5170,...,472.0105,87.2054,1.7079,0.4718,41.1692,4.1999,4.2971,5.5643,8.4978,0.2997
TCGA-C8-A274-01A-11R-A16F-07,3.6472,0.3817,10.6594,2.7049,0.8136,13.6743,32.6223,12.3757,0.0000,91.8483,...,498.0916,100.3860,1.5072,0.0411,109.9864,7.8364,9.3964,10.3968,27.2978,0.2590


In [ ]:
#Added
variances = rna_tf.var()
sorted_variances = variances.sort_values(ascending=False)
top_rna_tf_names = sorted_variances.head(round(len(sorted_variances)*0.25)).index.tolist()
len(top_rna_tf_names)

690

In [ ]:
#Added
rna_tf = rna_tf[top_rna_tf_names]
rna_tf

,ENSG00000163220,ENSG00000143546,ENSG00000168878,ENSG00000211899,ENSG00000171401,ENSG00000120885,ENSG00000012223,ENSG00000196126,ENSG00000148303,ENSG00000100219,...,ENSG00000166949,ENSG00000079432,ENSG00000168283,ENSG00000115993,ENSG00000132109,ENSG00000169740,ENSG00000150347,ENSG00000111424,ENSG00000127616,ENSG00000136848
TCGA-D8-A146-01A-31R-A115-07,52.4808,9.4738,0.0321,334.2779,0.0679,329.8676,1129.1205,1195.6175,1401.7846,2694.5612,...,26.2725,53.5961,74.4540,68.6399,37.0117,65.4042,51.2438,35.8323,41.6272,27.3698
TCGA-AQ-A0Y5-01A-11R-A14M-07,2059.8340,486.9378,0.0302,0.5309,0.0639,1303.9199,41.3800,671.9958,1177.3698,4417.9372,...,14.4116,32.8697,76.6827,30.9150,21.4305,43.4970,39.3726,46.4106,46.5177,20.8468
TCGA-C8-A274-01A-11R-A16F-07,8682.2141,470.2573,0.0783,5.8758,0.0474,100.0401,9714.9638,474.6179,1612.2658,3307.9336,...,23.8669,45.1699,101.4895,70.4358,41.5768,96.9117,42.7323,35.0784,40.0692,18.9195
TCGA-BH-A0BD-01A-11R-A034-07,39.1997,9.5351,0.0566,403.6612,0.1398,3222.1057,1406.1772,1350.2580,1395.6539,3757.3782,...,30.9661,50.7800,83.1212,22.2820,68.1141,56.6358,36.2294,46.4373,38.7476,33.5867
TCGA-B6-A1KC-01B-11R-A157-07,1.8851,0.7633,0.0195,1.2401,0.0000,67.7256,0.8023,200.0585,548.0068,5600.3306,...,24.8349,18.2960,149.0432,34.6186,6.4918,54.2536,5.1788,7.6640,18.0893,10.6423
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-CV-7432-01A-11R-2132-07,19601.1862,7916.3651,0.0403,0.8843,154.9594,98.5141,0.1797,251.8738,2712.7872,69.7744,...,30.0100,40.2437,8.5704,12.1657,71.2523,2.0020,13.1074,35.6711,54.4912,87.1862
TCGA-IQ-7630-01A-11R-2081-07,22776.8807,6616.2773,0.0789,158.1232,20.3091,5.3441,0.0581,557.1760,583.1945,46.9703,...,72.7339,38.8959,22.4839,12.3955,111.4019,23.5763,22.1273,61.9145,43.6286,79.8336
TCGA-H7-A6C4-11A-21R-A466-07,28447.2937,20788.6111,0.0383,1.2057,9197.3727,49.6838,5.0109,294.9310,1139.6156,37.9788,...,20.1982,9.9467,7.7114,7.2385,9.7606,23.8242,15.9726,9.7548,11.7173,17.1698
TCGA-H7-8501-01A-11R-2403-07,7688.1677,2472.5636,0.4315,71.4626,144.3971,16.2572,0.6175,2782.4379,1941.9144,146.3322,...,16.9710,17.5704,25.7145,16.4917,176.1337,41.0996,10.1118,43.7086,18.8876,33.5202


In [ ]:
transfer_data_full = pd.read_csv('drive/My Drive/Colab Notebooks/Deep_learning_metabolomics_prediction/gene_expression_benedetti_dataset.csv', index_col = 0)
transfer_data_full.head(3)

,ENSG00000121410,ENSG00000175899,ENSG00000291190,ENSG00000171428,ENSG00000156006,ENSG00000253937,ENSG00000196136,ENSG00000114771,ENSG00000127837,ENSG00000129673,...,ENSG00000207734,ENSG00000207735,ENSG00000207925,ENSG00000283330,ENSG00000207699,ENSG00000291012,ENSG00000274847,ENSG00000261509,ENSG00000276941,ENSG00000183385
MSKC-00397,1.304026,320.190822,4.301926,8.314472,0.964676,NaN,123.587134,0.210955,180.621121,8.316147,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MSKC-00399,1.928762,599.699496,1.251719,10.039836,3.234161,NaN,3.693906,0.000000,227.394412,0.349771,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MSKC-00400,0.869958,759.630310,1.620558,5.763073,0.369455,NaN,0.519229,0.242377,219.065986,0.769822,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
common_tf_transfer = list(set(rna_tf.columns.to_list()).intersection(transfer_data_full.columns.to_list()))
print(common_tf_transfer)

['ENSG00000164930', 'ENSG00000074219', 'ENSG00000064961', 'ENSG00000077150', 'ENSG00000169136', 'ENSG00000197905', 'ENSG00000124226', 'ENSG00000159184', 'ENSG00000137166', 'ENSG00000108654', 'ENSG00000168496', 'ENSG00000055130', 'ENSG00000163132', 'ENSG00000124575', 'ENSG00000165672', 'ENSG00000107447', 'ENSG00000126778', 'ENSG00000113068', 'ENSG00000112715', 'ENSG00000163558', 'ENSG00000215301', 'ENSG00000087191', 'ENSG00000135111', 'ENSG00000111206', 'ENSG00000168283', 'ENSG00000117318', 'ENSG00000213676', 'ENSG00000172534', 'ENSG00000144485', 'ENSG00000114554', 'ENSG00000115641', 'ENSG00000165030', 'ENSG00000106261', 'ENSG00000196843', 'ENSG00000196975', 'ENSG00000106483', 'ENSG00000136997', 'ENSG00000135363', 'ENSG00000103994', 'ENSG00000173253', 'ENSG00000180818', 'ENSG00000197780', 'ENSG00000204209', 'ENSG00000157933', 'ENSG00000118418', 'ENSG00000185650', 'ENSG00000164920', 'ENSG00000170608', 'ENSG00000082175', 'ENSG00000137310', 'ENSG00000129654', 'ENSG00000177700', 'ENSG000001

In [ ]:
transfer_tf = transfer_data_full[common_tf_transfer]
transfer_tf.head(3)

,ENSG00000164930,ENSG00000074219,ENSG00000064961,ENSG00000077150,ENSG00000169136,ENSG00000197905,ENSG00000124226,ENSG00000159184,ENSG00000137166,ENSG00000108654,...,ENSG00000124160,ENSG00000123358,ENSG00000004399,ENSG00000125520,ENSG00000101126,ENSG00000147889,ENSG00000116670,ENSG00000144802,ENSG00000075891,ENSG00000110244
MSKC-00397,14.529881,29.841657,115.192800,79.471610,65.262479,28.886094,137.021485,1.367006,14.180497,390.247090,...,36.296144,42.470505,72.975298,63.532046,36.014415,17.282096,21.648789,29.726232,3.186778,0.0
MSKC-00399,24.450902,40.405652,85.443224,28.476253,30.171780,15.768220,118.166876,0.041264,22.427154,470.020002,...,44.311957,495.693384,43.022897,175.405866,47.074926,0.120574,28.120433,18.446672,81.676243,0.0
MSKC-00400,12.766091,25.769314,72.159172,69.764160,15.898116,23.326659,102.403094,0.000000,9.814876,661.204682,...,49.922937,594.446477,98.383284,211.376486,48.502388,5.572871,17.698377,7.095124,79.121663,0.0


In [ ]:
rna_tf_transfer_tf_common = list(set(rna_tf.columns).intersection(transfer_tf.columns))
print(rna_tf_transfer_tf_common)

['ENSG00000164930', 'ENSG00000064961', 'ENSG00000074219', 'ENSG00000077150', 'ENSG00000169136', 'ENSG00000197905', 'ENSG00000124226', 'ENSG00000159184', 'ENSG00000137166', 'ENSG00000108654', 'ENSG00000168496', 'ENSG00000055130', 'ENSG00000163132', 'ENSG00000124575', 'ENSG00000165672', 'ENSG00000107447', 'ENSG00000126778', 'ENSG00000113068', 'ENSG00000112715', 'ENSG00000163558', 'ENSG00000215301', 'ENSG00000087191', 'ENSG00000135111', 'ENSG00000111206', 'ENSG00000168283', 'ENSG00000117318', 'ENSG00000213676', 'ENSG00000172534', 'ENSG00000144485', 'ENSG00000114554', 'ENSG00000115641', 'ENSG00000165030', 'ENSG00000106261', 'ENSG00000196843', 'ENSG00000196975', 'ENSG00000106483', 'ENSG00000136997', 'ENSG00000135363', 'ENSG00000103994', 'ENSG00000173253', 'ENSG00000180818', 'ENSG00000197780', 'ENSG00000204209', 'ENSG00000157933', 'ENSG00000118418', 'ENSG00000185650', 'ENSG00000164920', 'ENSG00000170608', 'ENSG00000082175', 'ENSG00000137310', 'ENSG00000129654', 'ENSG00000177700', 'ENSG000001

In [ ]:
rna_tf = rna_tf[rna_tf_transfer_tf_common]
transfer_tf = transfer_tf[rna_tf_transfer_tf_common]
print(rna_tf.shape)
print(transfer_tf.shape)

(10668, 682)
(443, 682)


In [ ]:
rna_tf.head(3)

,ENSG00000164930,ENSG00000064961,ENSG00000074219,ENSG00000077150,ENSG00000169136,ENSG00000197905,ENSG00000124226,ENSG00000159184,ENSG00000137166,ENSG00000108654,...,ENSG00000124160,ENSG00000123358,ENSG00000004399,ENSG00000125520,ENSG00000101126,ENSG00000147889,ENSG00000116670,ENSG00000144802,ENSG00000075891,ENSG00000096696
TCGA-D8-A146-01A-31R-A115-07,40.8331,99.4066,71.2573,45.3398,51.0103,16.0300,81.6687,0.0868,52.9615,324.0792,...,100.0648,46.4742,66.9054,118.9051,101.2596,7.8932,25.0373,55.3425,1.4379,324.6579
TCGA-AQ-A0Y5-01A-11R-A14M-07,53.8007,64.8461,27.3517,25.2578,50.7870,26.1682,45.5170,2.3900,77.1110,206.8882,...,87.2054,20.1423,49.2686,41.1692,107.5493,4.4901,16.6400,5.3116,1.8502,280.7401
TCGA-C8-A274-01A-11R-A16F-07,10.6594,67.3339,141.2521,44.0761,27.5640,9.1795,91.8483,15.0755,55.9633,285.5990,...,100.3860,3.4302,56.1876,109.9864,148.0278,0.8051,20.9096,94.5375,3.7159,218.6022


In [ ]:
max_var_col in tf['Ensembl ID'].to_list()

False

In [ ]:
rna_tf = pd.concat([rna[[max_var_col]], rna_tf], axis = 'columns')
rna_tf.head(3)

,ENSG00000211592,ENSG00000164930,ENSG00000064961,ENSG00000074219,ENSG00000077150,ENSG00000169136,ENSG00000197905,ENSG00000124226,ENSG00000159184,ENSG00000137166,...,ENSG00000124160,ENSG00000123358,ENSG00000004399,ENSG00000125520,ENSG00000101126,ENSG00000147889,ENSG00000116670,ENSG00000144802,ENSG00000075891,ENSG00000096696
TCGA-D8-A146-01A-31R-A115-07,7654.8063,40.8331,99.4066,71.2573,45.3398,51.0103,16.0300,81.6687,0.0868,52.9615,...,100.0648,46.4742,66.9054,118.9051,101.2596,7.8932,25.0373,55.3425,1.4379,324.6579
TCGA-AQ-A0Y5-01A-11R-A14M-07,10.8013,53.8007,64.8461,27.3517,25.2578,50.7870,26.1682,45.5170,2.3900,77.1110,...,87.2054,20.1423,49.2686,41.1692,107.5493,4.4901,16.6400,5.3116,1.8502,280.7401
TCGA-C8-A274-01A-11R-A16F-07,216.4474,10.6594,67.3339,141.2521,44.0761,27.5640,9.1795,91.8483,15.0755,55.9633,...,100.3860,3.4302,56.1876,109.9864,148.0278,0.8051,20.9096,94.5375,3.7159,218.6022


In [ ]:
transfer_tf.head(3)

,ENSG00000164930,ENSG00000064961,ENSG00000074219,ENSG00000077150,ENSG00000169136,ENSG00000197905,ENSG00000124226,ENSG00000159184,ENSG00000137166,ENSG00000108654,...,ENSG00000124160,ENSG00000123358,ENSG00000004399,ENSG00000125520,ENSG00000101126,ENSG00000147889,ENSG00000116670,ENSG00000144802,ENSG00000075891,ENSG00000096696
MSKC-00397,14.529881,115.192800,29.841657,79.471610,65.262479,28.886094,137.021485,1.367006,14.180497,390.247090,...,36.296144,42.470505,72.975298,63.532046,36.014415,17.282096,21.648789,29.726232,3.186778,13.814659
MSKC-00399,24.450902,85.443224,40.405652,28.476253,30.171780,15.768220,118.166876,0.041264,22.427154,470.020002,...,44.311957,495.693384,43.022897,175.405866,47.074926,0.120574,28.120433,18.446672,81.676243,25.055237
MSKC-00400,12.766091,72.159172,25.769314,69.764160,15.898116,23.326659,102.403094,0.000000,9.814876,661.204682,...,49.922937,594.446477,98.383284,211.376486,48.502388,5.572871,17.698377,7.095124,79.121663,0.150022


In [ ]:
rna_tf.to_csv('drive/My Drive/Colab Notebooks/Deep_learning_metabolomics_prediction/TCGA_Pan_rna_tf.csv', index = True)
transfer_tf.to_csv('drive/My Drive/Colab Notebooks/Deep_learning_metabolomics_prediction/transfer_data_tf.csv', index = True)

#Merge data

In [ ]:
merged = pd.concat([rna_tf, rna_met_annot.drop(labels = ['sample_type', 'ENSG00000211592'], axis = 'columns')], axis = 'columns')
merged

,ENSG00000211592,ENSG00000164930,ENSG00000064961,ENSG00000074219,ENSG00000077150,ENSG00000169136,ENSG00000197905,ENSG00000124226,ENSG00000159184,ENSG00000137166,...,ENSG00000205923,ENSG00000183747,ENSG00000066813,ENSG00000182601,ENSG00000156885,ENSG00000182156,ENSG00000166948,ENSG00000197838,ENSG00000167751,ENSG00000182415
TCGA-D8-A146-01A-31R-A115-07,7654.8063,40.8331,99.4066,71.2573,45.3398,51.0103,16.0300,81.6687,0.0868,52.9615,...,0.0,0.0402,0.0162,0.1149,0.1394,0.0562,0.0000,0.0749,0.0831,0.0
TCGA-AQ-A0Y5-01A-11R-A14M-07,10.8013,53.8007,64.8461,27.3517,25.2578,50.7870,26.1682,45.5170,2.3900,77.1110,...,0.0,0.0000,0.0152,0.0902,0.0000,0.0000,0.0000,0.0000,0.1173,0.0
TCGA-C8-A274-01A-11R-A16F-07,216.4474,10.6594,67.3339,141.2521,44.0761,27.5640,9.1795,91.8483,15.0755,55.9633,...,0.0,0.0000,0.0056,0.0000,0.0000,0.0131,0.0000,0.6273,0.0290,0.0
TCGA-BH-A0BD-01A-11R-A034-07,6638.7969,75.4110,64.3758,83.0089,64.5350,47.0632,26.5756,80.8786,0.0574,49.2848,...,0.0,0.0142,0.0071,0.1014,0.1230,0.0000,0.0000,2.2483,0.0733,0.0
TCGA-B6-A1KC-01B-11R-A157-07,74.1970,60.7607,37.1114,3.6246,14.7749,19.7951,10.4981,44.0269,0.0527,55.8907,...,0.0,0.0049,0.0000,0.0232,0.1692,0.0000,0.0348,0.0455,0.0126,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-CV-7432-01A-11R-2132-07,200.9838,67.9835,58.9956,20.5237,47.8842,23.6082,73.0152,48.1936,0.0000,27.6437,...,0.0,0.0000,0.0000,0.0641,0.0000,0.0000,0.0000,0.0000,0.0000,0.0
TCGA-IQ-7630-01A-11R-2081-07,139.9574,105.8413,64.0444,38.7878,58.5238,31.5900,40.3814,73.7803,0.0000,29.2523,...,0.0,0.0000,0.0057,0.0000,0.5877,0.2762,0.0202,0.1053,0.0438,0.0
TCGA-H7-A6C4-11A-21R-A466-07,268.1355,8.7504,16.7714,4.9647,20.8897,12.2632,36.1594,16.8386,0.0173,3.4598,...,0.0,0.0000,0.0000,0.7466,500.2332,0.0000,2.8753,0.0000,0.0000,0.0
TCGA-H7-8501-01A-11R-2403-07,5129.2628,105.6087,59.0624,9.0926,41.1974,16.3537,138.3765,131.9729,0.0000,15.1099,...,0.0,0.0000,0.0121,0.0286,0.2083,0.0839,0.0429,0.0560,0.0000,0.0


In [ ]:
merged_transfer = pd.concat([transfer_tf, transfer_data_met], axis = 'columns')
merged_transfer

,ENSG00000164930,ENSG00000064961,ENSG00000074219,ENSG00000077150,ENSG00000169136,ENSG00000197905,ENSG00000124226,ENSG00000159184,ENSG00000137166,ENSG00000108654,...,ENSG00000205923,ENSG00000183747,ENSG00000066813,ENSG00000182601,ENSG00000156885,ENSG00000182156,ENSG00000166948,ENSG00000197838,ENSG00000167751,ENSG00000182415
MSKC-00397,14.529881,115.192800,29.841657,79.471610,65.262479,28.886094,137.021485,1.367006,14.180497,390.247090,...,19.356151,5.898651,8.225321,0.026444,0.384994,2.161147,0.0,0.0,0.127099,0.0
MSKC-00399,24.450902,85.443224,40.405652,28.476253,30.171780,15.768220,118.166876,0.041264,22.427154,470.020002,...,23.797005,173.680890,235.708998,0.000000,4.840216,4.216679,0.0,0.0,0.000000,0.0
MSKC-00400,12.766091,72.159172,25.769314,69.764160,15.898116,23.326659,102.403094,0.000000,9.814876,661.204682,...,23.372477,534.686682,411.479407,0.000000,0.000000,0.316985,0.0,0.0,0.000000,0.0
MSKC-00404,18.893303,73.005636,40.702548,69.259393,53.030034,17.373790,97.193476,0.157535,14.712511,552.384331,...,12.879372,2.846888,2.450864,0.000000,0.181166,0.043275,0.0,0.0,0.119617,0.0
MSKC-00405,33.396663,71.878100,41.783107,54.901434,51.195302,27.794177,108.183866,4.507236,18.594852,612.888369,...,13.380325,42.575867,43.357611,0.000000,0.712212,0.623798,0.0,0.0,0.031350,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
PT-P44H,1.135020,5.492578,0.235548,1.823922,6.448137,0.080267,10.407573,0.000000,1.858637,189.931256,...,0.000000,0.007620,0.000000,0.284963,0.263497,0.016270,0.0,0.0,0.004840,0.0
PT-R55F,1.584404,4.856074,0.378914,3.443551,4.386524,0.173131,10.427795,0.000000,1.560770,211.971716,...,0.000000,0.006537,0.008831,0.192258,0.508649,0.010469,0.0,0.0,0.000000,0.0
PT-RN5K,1.645791,8.187012,0.913826,9.650913,4.019023,0.255058,11.193488,0.000000,2.737795,217.249014,...,0.000000,0.005306,0.000000,0.375665,0.688057,0.059481,0.0,0.0,0.015166,0.0
PT-UTHO,1.466925,9.055514,0.616683,5.029287,3.301576,0.098048,15.217994,0.000000,1.740849,360.455461,...,0.000000,0.000000,0.000000,0.399252,0.000000,0.117592,0.0,0.0,0.000000,0.0


In [ ]:
common_merged = list(set(merged.columns).intersection(merged_transfer.columns))
len(common_merged)

3494

In [ ]:
merged_common = merged[common_merged]
merged_transfer_common = merged_transfer[common_merged]

merged = pd.concat([merged[[max_var_col]], merged_common], axis = 'columns')
merged_transfer = merged_transfer_common

In [ ]:
merged.head(3)

,ENSG00000211592,ENSG00000164930,ENSG00000197594,ENSG00000124226,ENSG00000159184,ENSG00000198668,ENSG00000179142,ENSG00000007350,ENSG00000129744,ENSG00000170035,...,ENSG00000034677,ENSG00000135241,ENSG00000083123,ENSG00000136247,ENSG00000095303,ENSG00000130717,ENSG00000122643,ENSG00000141959,ENSG00000184307,ENSG00000115425
TCGA-D8-A146-01A-31R-A115-07,7654.8063,40.8331,48.1403,81.6687,0.0868,174.3412,0.0,0.0748,0.7642,31.8422,...,58.7566,42.2553,24.4281,30.9430,19.7246,52.3589,26.8390,68.9754,5.9405,8.6663
TCGA-AQ-A0Y5-01A-11R-A14M-07,10.8013,53.8007,46.3698,45.5170,2.3900,223.5212,0.0,0.0000,0.1269,27.1147,...,31.5056,35.0713,23.7728,26.4577,19.7589,27.6158,24.1808,79.0960,5.6007,6.4292
TCGA-C8-A274-01A-11R-A16F-07,216.4474,10.6594,86.4358,91.8483,15.0755,179.7067,0.0,0.0261,1.0661,43.3071,...,60.3827,26.8141,22.2365,11.0825,10.2192,40.1886,17.6016,52.4967,12.0377,23.0999


In [ ]:
keep = merged[['ENSG00000211592']].iloc[:,1]
merged = merged.drop(labels = ['ENSG00000211592'], axis = 'columns')
merged = pd.concat([keep, merged], axis = 'columns')
merged.head(3)

,ENSG00000211592,ENSG00000164930,ENSG00000197594,ENSG00000124226,ENSG00000159184,ENSG00000198668,ENSG00000179142,ENSG00000007350,ENSG00000129744,ENSG00000170035,...,ENSG00000034677,ENSG00000135241,ENSG00000083123,ENSG00000136247,ENSG00000095303,ENSG00000130717,ENSG00000122643,ENSG00000141959,ENSG00000184307,ENSG00000115425
TCGA-D8-A146-01A-31R-A115-07,7654.8063,40.8331,48.1403,81.6687,0.0868,174.3412,0.0,0.0748,0.7642,31.8422,...,58.7566,42.2553,24.4281,30.9430,19.7246,52.3589,26.8390,68.9754,5.9405,8.6663
TCGA-AQ-A0Y5-01A-11R-A14M-07,10.8013,53.8007,46.3698,45.5170,2.3900,223.5212,0.0,0.0000,0.1269,27.1147,...,31.5056,35.0713,23.7728,26.4577,19.7589,27.6158,24.1808,79.0960,5.6007,6.4292
TCGA-C8-A274-01A-11R-A16F-07,216.4474,10.6594,86.4358,91.8483,15.0755,179.7067,0.0,0.0261,1.0661,43.3071,...,60.3827,26.8141,22.2365,11.0825,10.2192,40.1886,17.6016,52.4967,12.0377,23.0999


In [ ]:
merged_transfer.head(3)

,ENSG00000164930,ENSG00000197594,ENSG00000124226,ENSG00000159184,ENSG00000198668,ENSG00000179142,ENSG00000007350,ENSG00000129744,ENSG00000170035,ENSG00000170950,...,ENSG00000034677,ENSG00000135241,ENSG00000083123,ENSG00000136247,ENSG00000095303,ENSG00000130717,ENSG00000122643,ENSG00000141959,ENSG00000184307,ENSG00000115425
MSKC-00397,14.529881,9.240603,137.021485,1.367006,194.934404,0.0,0.146082,0.115708,67.298137,0.0,...,40.706703,18.973572,6.662711,64.911143,43.204901,25.409572,58.497425,103.255543,1.397247,9.702442
MSKC-00399,24.450902,6.571562,118.166876,0.041264,344.872173,0.0,0.302495,0.000000,110.065381,0.0,...,56.435361,30.334806,35.942656,102.512775,3.612696,55.110532,33.836278,138.506963,7.439917,55.062394
MSKC-00400,12.766091,0.371475,102.403094,0.000000,277.825765,0.0,0.067137,0.066472,126.381236,0.0,...,52.821845,26.818338,14.078094,155.775369,6.503317,57.705789,61.026194,194.356737,4.903670,56.886399


In [ ]:
merged.to_csv('drive/My Drive/Colab Notebooks/Deep_learning_metabolomics_prediction/TCGA_Pan_rna_combination.csv', index = True)
merged_transfer.to_csv('drive/My Drive/Colab Notebooks/Deep_learning_metabolomics_prediction/transfer_data_combination.csv', index = True)